# 00 · Destil·lació: d'un Qwen gran a un Qwen petit
### *Un analitzador sintàctic del català que cap a la butxaca*

---


**La tasca.** Entra una oració simple en català i en surt l'anàlisi sintàctica escolar en JSON: subjecte, nucli verbal, CD, CI, atribut, CPRED, CRV, CAG, CC…

1. **El petit**, tal com surt de fàbrica → no se'n surt
2. **El professor** escriu oracions amb una estructura demanada i les analitza *a cegues*
3. **Filtre**: validació automàtica + coincidència amb l'estructura demanada
4. **Destil·lació**: fine-tuning del petit amb les dades bones
5. **Avaluació**: petit vs destil·lat vs professor, sobre 30 oracions anotades a mà

Tot es desa a Google Drive (`MyDrive/destillacio_sintaxi`): si el dataset o el model ja hi són, el notebook els reutilitza i se salta els passos lents.

# 01 · Preparació
### *GPU, Google Drive i opcions*

---

In [ ]:
#@title Opcions
REGENERAR_FRASES   = False   # True → el professor torna a escriure les oracions
REGENERAR_ANOTACIO = False   # True → el professor torna a analitzar-les
REENTRENAR         = False   # True → torna a entrenar l'alumne
PETICIONS          = 500     # × 8 oracions per petició
PENSAR_PROFESSOR   = True    # el professor raona (thinking) abans de respondre: més lent, més fiable

import os, json, glob, random, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/destillacio_sintaxi'
os.makedirs(BASE, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
print('Fitxers ja desats:', sorted(os.listdir(BASE)))

### Instal·lació

In [ ]:
#@title Instal·la vLLM, TRL i companyia
!pip install -q -U vllm trl peft datasets hf_transfer
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# 02 · Les regles del joc
### *El mateix prompt per a tothom, validació automàtica i 30 oracions anotades a mà*

---

Les cel·les següents escriuen el codi compartit. Al **prompt** hi ha les instruccions i un sol exemple; la **validació** comprova que cada fragment sigui literal de l'oració, que no se solapin i que l'estructura sigui coherent (nominal ⇔ atribut, CAG ⇔ passiva…).

In [ ]:
#@title Escriu common.py
open('/content/common.py', 'w', encoding='utf-8').write(r'''# common.py — codi compartit: prompt, validació, mètriques i referència manual
import json, re, unicodedata, html

FUNCIONS = ["CD", "CI", "ATRIBUT", "CPRED", "CRV", "CAG",
            "CC_TEMPS", "CC_LLOC", "CC_MANERA", "CC_QUANTITAT",
            "CC_COMPANYIA", "CC_INSTRUMENT", "CC_CAUSA", "CC_FINALITAT"]

SISTEMA = """Ets un analitzador sintàctic del català. Analitza l'oració simple que et donin segons la gramàtica escolar catalana i respon NOMÉS amb un objecte JSON (sense cap text abans ni després) amb aquest format:
{"subjecte": {"text": <text literal del subjecte, o null si no n'hi ha>, "nucli": <nucli del subjecte, o null>, "eliptic": <true si el subjecte està omès>},
 "predicat": {"tipus": "verbal" o "nominal", "nucli": <forma verbal completa, amb auxiliars i perífrasis>,
              "complements": [{"funcio": <FUNCIÓ>, "text": <text literal>}]},
 "veu": "activa" o "passiva",
 "impersonal": <true o false>}

FUNCIÓ ha de ser una d'aquestes: CD, CI, ATRIBUT, CPRED, CRV, CAG, CC_TEMPS, CC_LLOC, CC_MANERA, CC_QUANTITAT, CC_COMPANYIA, CC_INSTRUMENT, CC_CAUSA, CC_FINALITAT.

Regles:
- Tots els textos han de ser fragments literals de l'oració, sense la puntuació final.
- Els pronoms febles (em, et, el, la, li, els, ens, us, hi, en...) són complements i s'etiqueten amb la seva funció.
- En «haver-hi», el «hi» forma part del nucli verbal («hi ha»).
- Predicat nominal: verb copulatiu (ser, estar, semblar) + ATRIBUT.
- Oració impersonal: subjecte amb text null, nucli null i eliptic false.
- Subjecte el·líptic: text null, nucli null i eliptic true.

Exemple — Oració: Ahir la Marta va comprar pa al forn.
{"subjecte": {"text": "la Marta", "nucli": "Marta", "eliptic": false}, "predicat": {"tipus": "verbal", "nucli": "va comprar", "complements": [{"funcio": "CC_TEMPS", "text": "Ahir"}, {"funcio": "CD", "text": "pa"}, {"funcio": "CC_LLOC", "text": "al forn"}]}, "veu": "activa", "impersonal": false}"""


def render(tok, frase, thinking=False, curt=False):
    # curt=True: només l'oració, sense instruccions ni exemple (així respon el model destil·lat)
    msgs = [{"role": "user", "content": frase}] if curt else \
           [{"role": "system", "content": SISTEMA}, {"role": "user", "content": f"Oració: {frase}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=thinking)


def extreu_json(text):
    if text is None:
        return None
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[1]
    s, e = text.find("{"), text.rfind("}")
    if s < 0 or e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except Exception:
        return None


def _base(t):
    return unicodedata.normalize("NFC", t).replace("’", "'").replace("`", "'").lower()


def norm(t):
    if t is None:
        return None
    t = re.sub(r"\s+", " ", _base(str(t))).strip()
    return t.strip(" .,;:!?¡¿\"«»")


def _ocurrencies(frase_n, frag):
    pre = "" if frag.startswith("'") else r"(?<!\w)"
    post = "" if frag.endswith("'") else r"(?!\w)"
    return [(m.start(), m.end()) for m in re.finditer(pre + re.escape(frag) + post, frase_n)]


def assigna_spans(frase, fragments):
    """Troba una posició no solapada per a cada fragment. Retorna llista de spans o None."""
    fn = re.sub(r"\s", " ", _base(frase))
    occ = [_ocurrencies(fn, norm(f)) if norm(f) else [] for f in fragments]
    if any(len(o) == 0 for o in occ):
        return None
    ordre = sorted(range(len(fragments)), key=lambda i: len(occ[i]))
    res = [None] * len(fragments)

    def bt(k, usats):
        if k == len(ordre):
            return True
        i = ordre[k]
        for s, e in occ[i]:
            if all(e <= a or s >= b for a, b in usats):
                res[i] = (s, e)
                if bt(k + 1, usats + [(s, e)]):
                    return True
        return False
    return res if bt(0, []) else None


def valida(frase, a):
    """Comprovacions deterministes. Retorna (ok, motiu)."""
    try:
        if not isinstance(a, dict):
            return False, "no és un objecte"
        s, p = a["subjecte"], a["predicat"]
        if a["veu"] not in ("activa", "passiva"):
            return False, "veu"
        if not isinstance(a["impersonal"], bool) or not isinstance(s["eliptic"], bool):
            return False, "booleans"
        if p["tipus"] not in ("verbal", "nominal"):
            return False, "tipus predicat"
        if not p.get("nucli"):
            return False, "sense nucli verbal"
        comps = p["complements"]
        if not isinstance(comps, list):
            return False, "complements"
        frags = [p["nucli"]]
        for c in comps:
            if c.get("funcio") not in FUNCIONS:
                return False, f"funció desconeguda {c.get('funcio')}"
            if not c.get("text"):
                return False, "complement buit"
            frags.append(c["text"])
        if s["text"] is None:
            if s["nucli"] is not None:
                return False, "nucli sense subjecte"
            if a["impersonal"] == s["eliptic"]:
                return False, "sense subjecte: ha de ser impersonal XOR el·líptic"
        else:
            if a["impersonal"] or s["eliptic"]:
                return False, "subjecte explícit però marcat impersonal/el·líptic"
            if not s["nucli"] or norm(s["nucli"]) not in norm(s["text"]):
                return False, "nucli del subjecte fora del subjecte"
            frags.append(s["text"])
        fs = [c["funcio"] for c in comps]
        if (p["tipus"] == "nominal") != ("ATRIBUT" in fs):
            return False, "nominal ⇔ ATRIBUT"
        if "CAG" in fs and a["veu"] != "passiva":
            return False, "CAG sense passiva"
        if assigna_spans(frase, frags) is None:
            return False, "fragments no literals o solapats"
        return True, "ok"
    except Exception as e:
        return False, f"estructura ({type(e).__name__})"


def estructura(a):
    s = a["subjecte"]
    tipus_s = "impersonal" if a["impersonal"] else ("eliptic" if s["eliptic"] else "explicit")
    return {"subjecte": tipus_s, "tipus": a["predicat"]["tipus"], "veu": a["veu"],
            "funcions": sorted(set(c["funcio"] for c in a["predicat"]["complements"]))}


def _parells(a):
    return {(c["funcio"], norm(c["text"])) for c in a["predicat"]["complements"]}


def avalua(prediccions, referencies, frases):
    n = len(referencies)
    m = dict(json_valid=0, analisi_valida=0, subjecte=0, nucli_verbal=0, tipus_i_veu=0, exacte=0)
    tp = fp = fn = 0
    ltp = lfp = lfn = 0
    for pred, ref, fr in zip(prediccions, referencies, frases):
        rp = _parells(ref)
        rl = [f for f, _ in rp]
        if not isinstance(pred, dict):
            fn += len(rp); lfn += len(rl)
            continue
        m["json_valid"] += 1
        ok, _ = valida(fr, pred)
        m["analisi_valida"] += ok
        try:
            sp, sr = pred["subjecte"], ref["subjecte"]
            subj = norm(sp.get("text")) == norm(sr["text"]) and bool(sp.get("eliptic")) == sr["eliptic"]
            nuc = norm(pred["predicat"].get("nucli")) == norm(ref["predicat"]["nucli"])
            tv = pred["predicat"].get("tipus") == ref["predicat"]["tipus"] and pred.get("veu") == ref["veu"]
            pp = _parells(pred)
        except Exception:
            subj = nuc = tv = False
            pp = set()
        m["subjecte"] += subj; m["nucli_verbal"] += nuc; m["tipus_i_veu"] += tv
        m["exacte"] += subj and nuc and tv and pp == rp
        tp += len(pp & rp); fp += len(pp - rp); fn += len(rp - pp)
        pl = sorted(f for f, _ in pp)
        rl2 = sorted(rl)
        comu = sum(min(pl.count(x), rl2.count(x)) for x in set(pl) | set(rl2))
        ltp += comu; lfp += len(pl) - comu; lfn += len(rl2) - comu
    res = {k: round(100 * v / n, 1) for k, v in m.items()}
    f1 = lambda a, b, c: round(100 * 2 * a / max(1, 2 * a + b + c), 1)
    res["F1_complements"] = f1(tp, fp, fn)
    res["F1_funcions"] = f1(ltp, lfp, lfn)
    res["n"] = n
    return res


COLORS = {"SUBJ": "#4C78A8", "NV": "#E45756", "CD": "#F58518", "CI": "#54A24B", "ATRIBUT": "#B279A2",
          "CPRED": "#9D755D", "CRV": "#72B7B2", "CAG": "#FF9DA6"}


def mostra_html(frase, a):
    """Pinta l'oració amb els constituents marcats."""
    if not isinstance(a, dict):
        return f"<p>{html.escape(frase)} <b style='color:#c00'>→ sortida no analitzable</b></p>"
    try:
        etiquetes, frags = [], []
        if a["subjecte"]["text"]:
            etiquetes.append("SUBJ"); frags.append(a["subjecte"]["text"])
        etiquetes.append("NV"); frags.append(a["predicat"]["nucli"])
        for c in a["predicat"]["complements"]:
            etiquetes.append(c["funcio"]); frags.append(c["text"])
        spans = assigna_spans(frase, frags)
    except Exception:
        spans = None
    if spans is None:
        return (f"<p>{html.escape(frase)} <b style='color:#c00'>→ anàlisi incoherent</b><br>"
                f"<code>{html.escape(json.dumps(a, ensure_ascii=False))[:400]}</code></p>")
    out, pos = [], 0
    for (s, e), et in sorted(zip(spans, etiquetes)):
        out.append(html.escape(frase[pos:s]))
        col = COLORS.get(et, "#888")
        out.append(f"<span style='border-bottom:3px solid {col};padding:1px 2px'>{html.escape(frase[s:e])}"
                   f"<sub style='color:{col};font-weight:bold'>{et}</sub></span>")
        pos = e
    out.append(html.escape(frase[pos:]))
    extra = []
    if a["subjecte"]["eliptic"]: extra.append("subjecte el·líptic")
    if a["impersonal"]: extra.append("impersonal")
    extra.append(f"predicat {a['predicat']['tipus']}, veu {a['veu']}")
    return f"<p style='font-size:1.1em'>{''.join(out)} <i style='color:#666'>({'; '.join(extra)})</i></p>"


# ---------------------------------------------------------------------------
# Referència manual (30 oracions). Anotació feta a mà seguint la gramàtica
# escolar; és revisable — si hi discrepes, corregeix-la aquí.
# ---------------------------------------------------------------------------
def _A(subj, nucli_s, tipus, nv, comps, veu="activa", impersonal=False, eliptic=False):
    return {"subjecte": {"text": subj, "nucli": nucli_s, "eliptic": eliptic},
            "predicat": {"tipus": tipus, "nucli": nv,
                         "complements": [{"funcio": f, "text": t} for f, t in comps]},
            "veu": veu, "impersonal": impersonal}

REFERENCIA = [
    ("La Maria llegeix un llibre a la biblioteca.", _A("La Maria", "Maria", "verbal", "llegeix", [("CD", "un llibre"), ("CC_LLOC", "a la biblioteca")])),
    ("Els nens juguen al parc cada tarda.", _A("Els nens", "nens", "verbal", "juguen", [("CC_LLOC", "al parc"), ("CC_TEMPS", "cada tarda")])),
    ("El meu germà és metge.", _A("El meu germà", "germà", "nominal", "és", [("ATRIBUT", "metge")])),
    ("Demà anirem a la platja amb els cosins.", _A(None, None, "verbal", "anirem", [("CC_TEMPS", "Demà"), ("CC_LLOC", "a la platja"), ("CC_COMPANYIA", "amb els cosins")], eliptic=True)),
    ("L'àvia va regalar una bufanda a la Júlia.", _A("L'àvia", "àvia", "verbal", "va regalar", [("CD", "una bufanda"), ("CI", "a la Júlia")])),
    ("Li he enviat la carta aquest matí.", _A(None, None, "verbal", "he enviat", [("CI", "Li"), ("CD", "la carta"), ("CC_TEMPS", "aquest matí")], eliptic=True)),
    ("La casa va ser construïda pels avis.", _A("La casa", "casa", "verbal", "va ser construïda", [("CAG", "pels avis")], veu="passiva")),
    ("Plou molt a la muntanya.", _A(None, None, "verbal", "Plou", [("CC_QUANTITAT", "molt"), ("CC_LLOC", "a la muntanya")], impersonal=True)),
    ("Hi ha molta gent a la plaça.", _A(None, None, "verbal", "Hi ha", [("CD", "molta gent"), ("CC_LLOC", "a la plaça")], impersonal=True)),
    ("En Pere confia en els seus amics.", _A("En Pere", "Pere", "verbal", "confia", [("CRV", "en els seus amics")])),
    ("Els alumnes van arribar cansats a l'escola.", _A("Els alumnes", "alumnes", "verbal", "van arribar", [("CPRED", "cansats"), ("CC_LLOC", "a l'escola")])),
    ("Van nomenar la Laia delegada de classe.", _A(None, None, "verbal", "Van nomenar", [("CD", "la Laia"), ("CPRED", "delegada de classe")], eliptic=True)),
    ("La sopa està molt calenta.", _A("La sopa", "sopa", "nominal", "està", [("ATRIBUT", "molt calenta")])),
    ("El fuster talla la fusta amb una serra.", _A("El fuster", "fuster", "verbal", "talla", [("CD", "la fusta"), ("CC_INSTRUMENT", "amb una serra")])),
    ("Vam ajornar la festa per la pluja.", _A(None, None, "verbal", "Vam ajornar", [("CD", "la festa"), ("CC_CAUSA", "per la pluja")], eliptic=True)),
    ("Estudia cada dia per aprovar l'examen.", _A(None, None, "verbal", "Estudia", [("CC_TEMPS", "cada dia"), ("CC_FINALITAT", "per aprovar l'examen")], eliptic=True)),
    ("La Núria parla l'anglès perfectament.", _A("La Núria", "Núria", "verbal", "parla", [("CD", "l'anglès"), ("CC_MANERA", "perfectament")])),
    ("Els turistes mengen paella al port.", _A("Els turistes", "turistes", "verbal", "mengen", [("CD", "paella"), ("CC_LLOC", "al port")])),
    ("Aquest llibre m'agrada molt.", _A("Aquest llibre", "llibre", "verbal", "agrada", [("CI", "m'"), ("CC_QUANTITAT", "molt")])),
    ("La professora ens explica la lliçó amb paciència.", _A("La professora", "professora", "verbal", "explica", [("CI", "ens"), ("CD", "la lliçó"), ("CC_MANERA", "amb paciència")])),
    ("L'he vist al mercat.", _A(None, None, "verbal", "he vist", [("CD", "L'"), ("CC_LLOC", "al mercat")], eliptic=True)),
    ("Els veïns semblen contents.", _A("Els veïns", "veïns", "nominal", "semblen", [("ATRIBUT", "contents")])),
    ("El partit va ser guanyat per l'equip local.", _A("El partit", "partit", "verbal", "va ser guanyat", [("CAG", "per l'equip local")], veu="passiva")),
    ("Fa fred aquesta nit.", _A(None, None, "verbal", "Fa", [("CD", "fred"), ("CC_TEMPS", "aquesta nit")], impersonal=True)),
    ("La Marta pensa en les vacances.", _A("La Marta", "Marta", "verbal", "pensa", [("CRV", "en les vacances")])),
    ("El nen dorm tranquil al llit.", _A("El nen", "nen", "verbal", "dorm", [("CPRED", "tranquil"), ("CC_LLOC", "al llit")])),
    ("Ahir el meu pare va comprar pa i llet al forn.", _A("el meu pare", "pare", "verbal", "va comprar", [("CC_TEMPS", "Ahir"), ("CD", "pa i llet"), ("CC_LLOC", "al forn")])),
    ("Les flors del jardí són precioses.", _A("Les flors del jardí", "flors", "nominal", "són", [("ATRIBUT", "precioses")])),
    ("Escriurem una carta al director.", _A(None, None, "verbal", "Escriurem", [("CD", "una carta"), ("CI", "al director")], eliptic=True)),
    ("En Jordi treballa a Girona des de fa anys.", _A("En Jordi", "Jordi", "verbal", "treballa", [("CC_LLOC", "a Girona"), ("CC_TEMPS", "des de fa anys")])),
]
''')

In [ ]:
#@title Escriu infer.py
open('/content/infer.py', 'w', encoding='utf-8').write(r'''# infer.py — inferència per lots amb vLLM (s'executa en un procés a part per alliberar la GPU en acabar)
import argparse, json, sys, time
sys.path.insert(0, "/content")
from common import render, extreu_json

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--inp", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--thinking", type=int, default=0)
    ap.add_argument("--curt", type=int, default=0)
    ap.add_argument("--max_tokens", type=int, default=768)
    ap.add_argument("--temperature", type=float, default=0.0)
    ap.add_argument("--gpu_mem", type=float, default=0.90)
    ap.add_argument("--max_len", type=int, default=8192)
    args = ap.parse_args()

    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer

    rows = [json.loads(l) for l in open(args.inp, encoding="utf-8")]
    tok = AutoTokenizer.from_pretrained(args.model)
    prompts = [render(tok, r["frase"], bool(args.thinking), bool(args.curt)) for r in rows]

    kw = dict(model=args.model, gpu_memory_utilization=args.gpu_mem, max_model_len=args.max_len,
              enable_prefix_caching=True, max_num_seqs=256)
    try:
        llm = LLM(**kw, limit_mm_per_prompt={"image": 0, "video": 0})
    except Exception as e:
        print("Reintent sense limit_mm_per_prompt:", e)
        llm = LLM(**kw)

    sp = SamplingParams(temperature=args.temperature, top_p=0.95 if args.temperature > 0 else 1.0,
                        max_tokens=args.max_tokens)
    t0 = time.time()
    outs = llm.generate(prompts, sp)
    dt = time.time() - t0
    ntok = 0
    with open(args.out, "w", encoding="utf-8") as f:
        for r, o in zip(rows, outs):
            txt = o.outputs[0].text
            ntok += len(o.outputs[0].token_ids)
            r = dict(r, sortida=txt, analisi=extreu_json(txt), tokens=len(o.outputs[0].token_ids))
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    stats = {"model": args.model, "n": len(rows), "segons": round(dt, 1), "tokens_generats": ntok,
             "tokens_per_s": round(ntok / dt, 1), "ms_per_frase": round(1000 * dt / len(rows), 1),
             "tokens_prompt_mitjana": round(sum(len(o.prompt_token_ids) for o in outs) / len(outs), 1)}
    json.dump(stats, open(args.out + ".stats.json", "w"))
    print(stats)


if __name__ == "__main__":
    main()
''')

In [ ]:
#@title Escriu gen_frases.py
open('/content/gen_frases.py', 'w', encoding='utf-8').write(r'''# gen_frases.py — el professor escriu oracions catalanes amb una estructura sintàctica demanada
import argparse, json, random, re, sys, time
sys.path.insert(0, "/content")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--peticions", type=int, default=500)
    ap.add_argument("--per", type=int, default=8)
    ap.add_argument("--seed", type=int, default=1714)
    ap.add_argument("--gpu_mem", type=float, default=0.90)
    args = ap.parse_args()
    rng = random.Random(args.seed)

    TEMES = ["l'escola", "la cuina", "el mar i la pesca", "la muntanya", "la ciutat", "la feina a l'oficina",
             "la música", "la història de Catalunya", "la família", "els animals de companyia", "la tecnologia",
             "la festa major", "el mercat", "els trens i rodalies", "el bosc", "l'hospital", "la biblioteca",
             "la platja", "el teatre", "els castells", "la verema", "el pagès i l'hort", "el futbol",
             "un viatge", "el temps i el clima", "la política municipal", "un restaurant", "els jocs de taula",
             "la ciència", "una botiga de barri", "els avis", "el cinema", "un concert", "l'Ajuntament",
             "la bicicleta", "una mudança", "un museu", "el carnaval", "la neu", "una excursió escolar"]

    NOMS = {"CD": "complement directe (CD)", "CI": "complement indirecte (CI)", "ATRIBUT": "atribut",
            "CPRED": "complement predicatiu (CPRED)", "CRV": "complement de règim verbal (CRV)",
            "CAG": "complement agent (CAG)", "CC_TEMPS": "complement circumstancial de temps",
            "CC_LLOC": "complement circumstancial de lloc", "CC_MANERA": "complement circumstancial de manera",
            "CC_QUANTITAT": "complement circumstancial de quantitat",
            "CC_COMPANYIA": "complement circumstancial de companyia",
            "CC_INSTRUMENT": "complement circumstancial d'instrument",
            "CC_CAUSA": "complement circumstancial de causa", "CC_FINALITAT": "complement circumstancial de finalitat"}
    CC = ["CC_TEMPS"] * 4 + ["CC_LLOC"] * 4 + ["CC_MANERA"] * 3 + ["CC_QUANTITAT", "CC_COMPANYIA",
          "CC_INSTRUMENT", "CC_CAUSA", "CC_FINALITAT"]


    def ccs(k):
        out = []
        while len(out) < k:
            c = rng.choice(CC)
            if c not in out:
                out.append(c)
        return out


    def estructura_aleatoria():
        r = rng.random()
        subj = "explicit" if rng.random() < 0.65 else "eliptic"
        if r < 0.15:
            e = {"subjecte": subj, "tipus": "nominal", "veu": "activa", "funcions": ["ATRIBUT"] + ccs(rng.choice([0, 0, 1]))}
        elif r < 0.27:
            e = {"subjecte": "explicit", "tipus": "verbal", "veu": "passiva", "funcions": ["CAG"] + ccs(rng.choice([0, 1]))}
        elif r < 0.35:
            nucli = rng.choice([[], ["CD"]])
            e = {"subjecte": "impersonal", "tipus": "verbal", "veu": "activa", "funcions": nucli + ccs(rng.choice([1, 1, 2]))}
        else:
            nucli = rng.choice([[], ["CD"], ["CD"], ["CD"], ["CD", "CI"], ["CD", "CI"], ["CI"], ["CRV"], ["CPRED"], ["CD", "CPRED"]])
            k = rng.choice([0, 1, 1, 2]) if nucli else rng.choice([1, 2])
            e = {"subjecte": subj, "tipus": "verbal", "veu": "activa", "funcions": nucli + ccs(k)}
        e["funcions"] = sorted(set(e["funcions"]))
        e["febles"] = e["tipus"] == "verbal" and e["veu"] == "activa" and any(f in e["funcions"] for f in ("CD", "CI")) and rng.random() < 0.3
        return e


    def descriu(e):
        l = {"explicit": "- subjecte explícit", "eliptic": "- subjecte el·líptic (omès, no aparegui a l'oració)",
             "impersonal": "- oració impersonal, sense subjecte (verbs com ploure, nevar, fer fred, haver-hi...)"}[e["subjecte"]]
        l = [l]
        if e["tipus"] == "nominal":
            l.append("- predicat nominal amb verb copulatiu (ser, estar o semblar)")
        elif e["veu"] == "passiva":
            l.append("- veu passiva perifràstica (ser + participi)")
        else:
            l.append("- predicat verbal en veu activa")
        l += [f"- {NOMS[f]}" for f in e["funcions"]]
        if e["febles"]:
            l.append("- expressa el CD o el CI amb un pronom feble")
        return "\n".join(l)


    PLANTILLA = """Escriu {per} oracions simples en català sobre el tema «{tema}».
Han de ser oracions naturals, variades i correctes, amb una sola forma verbal (pot ser composta o perifràstica), sense subordinades ni coordinació d'oracions, i sense la paraula «no».
Cada oració ha de tenir EXACTAMENT aquests constituents, i cap altre complement:
{desc}
Respon només amb una llista JSON de {per} cadenes."""

    from vllm import LLM, SamplingParams
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(args.model)
    peticions = []
    for _ in range(args.peticions):
        e = estructura_aleatoria()
        tema = rng.choice(TEMES)
        msg = [{"role": "user", "content": PLANTILLA.format(per=args.per, tema=tema, desc=descriu(e))}]
        peticions.append((e, tema, tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)))

    kw = dict(model=args.model, gpu_memory_utilization=args.gpu_mem, max_model_len=4096, enable_prefix_caching=True, max_num_seqs=256)
    try:
        llm = LLM(**kw, limit_mm_per_prompt={"image": 0, "video": 0})
    except Exception as ex:
        print("Reintent sense limit_mm_per_prompt:", ex)
        llm = LLM(**kw)
    t0 = time.time()
    outs = llm.generate([p for _, _, p in peticions], SamplingParams(temperature=1.0, top_p=0.95, max_tokens=1200))
    vistes, n = set(), 0
    with open(args.out, "w", encoding="utf-8") as f:
        for (e, tema, _), o in zip(peticions, outs):
            t = o.outputs[0].text
            t = t.rsplit("</think>", 1)[-1]
            m = re.search(r"\[.*\]", t, re.S)
            try:
                frases = json.loads(m.group(0)) if m else []
            except Exception:
                frases = []
            for fr in frases:
                if not isinstance(fr, str):
                    continue
                fr = fr.strip()
                if 3 <= len(fr.split()) <= 25 and fr.lower() not in vistes:
                    vistes.add(fr.lower()); n += 1
                    f.write(json.dumps({"frase": fr, "estructura_demanada": e, "tema": tema}, ensure_ascii=False) + "\n")
    print({"frases": n, "peticions": len(peticions), "segons": round(time.time() - t0, 1)})


if __name__ == "__main__":
    main()
''')

In [ ]:
#@title Tria de models (el professor més gran que hi cap, l'alumne més petit)
import torch, importlib
sys.path.insert(0, '/content')
import common; importlib.reload(common)
from common import *
from huggingface_hub import HfApi
from IPython.display import HTML, display
import pandas as pd

api = HfApi()
def existeix(m):
    try: api.model_info(m); return True
    except Exception: return False

mem = torch.cuda.get_device_properties(0).total_memory / 1e9
cand_prof = (['Qwen/Qwen3.8-27B'] if mem > 75 else []) + \
            ['Qwen/Qwen3.8-27B-FP8', 'Qwen/Qwen3.6-27B-FP8', 'Qwen/Qwen3.5-27B-FP8'] + \
            (['Qwen/Qwen3.6-27B', 'Qwen/Qwen3.5-27B'] if mem > 75 else [])
cand_alum = ['Qwen/Qwen3.8-0.8B', 'Qwen/Qwen3.6-0.8B', 'Qwen/Qwen3.5-0.8B', 'Qwen/Qwen3-0.6B']
PROFESSOR = next(m for m in cand_prof if existeix(m))
ALUMNE    = next(m for m in cand_alum if existeix(m))
print(f'GPU: {mem:.0f} GB\nProfessor: {PROFESSOR}\nAlumne:    {ALUMNE}')

def corre(args):
    # vLLM en un procés a part: quan acaba, la GPU queda lliure
    p = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print(p.stderr[-6000:]); raise RuntimeError('ha fallat')

def llegeix(path): return [json.loads(l) for l in open(path, encoding='utf-8')]
def escriu(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')

REF_PATH = f'{BASE}/referencia_manual.jsonl'
escriu(REF_PATH, [{'frase': f, 'ref': a} for f, a in REFERENCIA])

# 03 · Abans de res: el petit, tal com ve
### *Mateix prompt, mateixes 30 oracions. A veure què fa.*

---

In [ ]:
#@title Executa el petit sobre les 30 oracions
corre(['/content/infer.py', '--model', ALUMNE, '--inp', REF_PATH, '--out', '/content/ref_alumne_base.jsonl', '--gpu_mem', '0.5'])
base_ref = llegeix('/content/ref_alumne_base.jsonl')
res_base = avalua([r['analisi'] for r in base_ref], [r['ref'] for r in base_ref], [r['frase'] for r in base_ref])
display(pd.DataFrame([res_base], index=[f'{ALUMNE} (base)']))

In [ ]:
#@title Uns quants exemples
for r in base_ref[:10]:
    display(HTML('<b>Referència:</b>' + mostra_html(r['frase'], r['ref']) + '<b>Petit base:</b>' + mostra_html(r['frase'], r['analisi'])))

# 04 · El professor fabrica el dataset
### *Primer escriu oracions amb l'estructura que li demanem; després les analitza sense saber-la*

---

El truc és que la **generació** i l'**anàlisi** són independents: si l'anàlisi a cegues coincideix amb l'estructura que havíem demanat, tenim dues opinions que concorden. Si no, la descartem.

In [ ]:
#@title 4a · Oracions amb estructura demanada
FRASES = f'{BASE}/frases.jsonl'
if REGENERAR_FRASES or not os.path.exists(FRASES):
    corre(['/content/gen_frases.py', '--model', PROFESSOR, '--out', FRASES, '--peticions', str(PETICIONS)])
frases = llegeix(FRASES)
print(len(frases), 'oracions'); print(*random.sample([f['frase'] for f in frases], 8), sep='\n')

In [ ]:
#@title 4b · El professor les analitza a cegues (sense saber l'estructura demanada)
ANOT = f'{BASE}/anotacio_professor.jsonl'
if REGENERAR_ANOTACIO or not os.path.exists(ANOT):
    corre(['/content/infer.py', '--model', PROFESSOR, '--inp', FRASES, '--out', ANOT,
           '--thinking', str(int(PENSAR_PROFESSOR)), '--max_tokens', '4096' if PENSAR_PROFESSOR else '768',
           '--temperature', '0.6' if PENSAR_PROFESSOR else '0', '--max_len', '8192'])
anot = llegeix(ANOT)
print(json.load(open(ANOT + '.stats.json')))

In [ ]:
#@title 4c · Filtre: validació determinista + coincidència amb l'estructura demanada
from collections import Counter
motius, bons = Counter(), []
for r in anot:
    a = r['analisi']
    if a is None: motius['JSON no vàlid'] += 1; continue
    ok, m = valida(r['frase'], a)
    if not ok: motius[m] += 1; continue
    dem = {k: v for k, v in r['estructura_demanada'].items() if k != 'febles'}
    if estructura(a) != dem: motius["no coincideix amb l'estructura demanada"] += 1; continue
    bons.append({'frase': r['frase'], 'analisi': a, 'estructura': dem})
print(f'{len(bons)}/{len(anot)} superen el filtre ({100*len(bons)/len(anot):.0f}%)')
display(pd.DataFrame(motius.most_common(), columns=['motiu de descart', 'n']))

# Treiem del dataset qualsevol oració de la referència manual
refs = {norm(f) for f, _ in REFERENCIA}
bons = [b for b in bons if norm(b['frase']) not in refs]
random.Random(0).shuffle(bons)
TEST, TRAIN = bons[:300], bons[300:]
escriu(f'{BASE}/train.jsonl', TRAIN); escriu(f'{BASE}/test.jsonl', TEST)
print('train', len(TRAIN), '· test', len(TEST))
display(pd.Series(Counter(f for b in TRAIN for f in b['estructura']['funcions'])).sort_values(ascending=False).to_frame('ocurrències al train'))

# 05 · El sostre
### *Quant encerta el professor en les 30 oracions anotades a mà?*

---

L'alumne aprèn del professor: el que el professor no sap, l'alumne tampoc no ho aprendrà.

In [ ]:
#@title El professor analitza les 30 oracions
REF_PROF = f'{BASE}/ref_professor.jsonl'
if not os.path.exists(REF_PROF):
    corre(['/content/infer.py', '--model', PROFESSOR, '--inp', REF_PATH, '--out', REF_PROF,
           '--thinking', str(int(PENSAR_PROFESSOR)), '--max_tokens', '4096', '--temperature', '0.6' if PENSAR_PROFESSOR else '0'])
prof_ref = llegeix(REF_PROF)
res_prof = avalua([r['analisi'] for r in prof_ref], [r['ref'] for r in prof_ref], [r['frase'] for r in prof_ref])
display(pd.DataFrame([res_base, res_prof], index=[f'{ALUMNE} (base)', PROFESSOR]))

# 06 · Destil·lació
### *Fine-tuning complet de l'alumne amb les anàlisis filtrades del professor*

---

Destil·lació per seqüència (SFT): l'alumne aprèn a donar la resposta final directament, **sense raonar**. El professor sí que ha raonat per arribar-hi.

I una cosa més: l'alumne s'entrena **sense prompt**. Només li donem l'oració; les instruccions, el format i l'exemple queden *dins dels pesos*. El model destil·lat respon directament amb el JSON.

In [ ]:
#@title Entrena (o recupera de Drive) l'alumne destil·lat
MODEL_OUT = f'{BASE}/alumne_destillat'
if REENTRENAR or not os.path.exists(f'{MODEL_OUT}/config.json'):
    from transformers import AutoTokenizer
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    tok = AutoTokenizer.from_pretrained(ALUMNE)
    FI = '<|im_end|>'
    def a_pc(rows):
        return Dataset.from_list([{'prompt': render(tok, r['frase'], False, curt=True),
                                   'completion': json.dumps(r['analisi'], ensure_ascii=False) + FI} for r in rows])
    ds_train, ds_eval = a_pc(TRAIN), a_pc(TEST[:100])
    print(ds_train[0]['prompt'][-200:], ds_train[0]['completion'])
    cfg = SFTConfig(output_dir='/content/sft', num_train_epochs=3, learning_rate=2e-5, lr_scheduler_type='cosine',
                    warmup_ratio=0.05, per_device_train_batch_size=16, gradient_accumulation_steps=1,
                    bf16=True, logging_steps=10, eval_strategy='epoch', save_strategy='no', report_to='none',
                    max_length=1536, completion_only_loss=True, gradient_checkpointing=True,
                    model_init_kwargs={'torch_dtype': torch.bfloat16})
    trainer = SFTTrainer(model=ALUMNE, args=cfg, train_dataset=ds_train, eval_dataset=ds_eval, processing_class=tok)
    trainer.train()
    trainer.save_model(MODEL_OUT); tok.save_pretrained(MODEL_OUT)
    # vLLM necessita també els fitxers de configuració originals (processador, plantilla...)
    from huggingface_hub import snapshot_download
    import shutil
    orig = snapshot_download(ALUMNE, allow_patterns=['*.json', '*.jinja', '*.txt'])
    for f in os.listdir(orig):
        if not os.path.exists(f'{MODEL_OUT}/{f}'): shutil.copy(f'{orig}/{f}', MODEL_OUT)
    pd.DataFrame(trainer.state.log_history).to_csv(f'{BASE}/log_entrenament.csv', index=False)
    del trainer; import gc; gc.collect(); torch.cuda.empty_cache()
print(sorted(os.listdir(MODEL_OUT)))

# 07 · Resultats
### *Petit base · petit destil·lat · professor*

---

In [ ]:
#@title Avalua els tres models
TEST_PATH = f'{BASE}/test_inp.jsonl'
escriu(TEST_PATH, [{'frase': r['frase'], 'ref': r['analisi']} for r in TEST])
for nom, model in [('base', ALUMNE), ('destillat', MODEL_OUT)]:
    for conj, path in [('ref', REF_PATH), ('test', TEST_PATH)]:
        out = f'/content/{conj}_alumne_{nom}.jsonl'
        if not os.path.exists(out) or nom == 'destillat':
            corre(['/content/infer.py', '--model', model, '--inp', path, '--out', out, '--gpu_mem', '0.5'] + (['--curt', '1'] if nom == 'destillat' else []))

files = {}
def taula(conj):
    rows, idx = [], []
    for nom, path in [(f'{ALUMNE} base', f'/content/{conj}_alumne_base.jsonl'),
                      ('alumne destil·lat', f'/content/{conj}_alumne_destillat.jsonl')] + \
                     ([(f'{PROFESSOR} (professor)', REF_PROF)] if conj == 'ref' else []):
        rs = llegeix(path); files[(conj, nom)] = rs
        r = avalua([x['analisi'] for x in rs], [x['ref'] for x in rs], [x['frase'] for x in rs])
        st = json.load(open(path + '.stats.json')); r['ms/frase'] = st['ms_per_frase']; r['tokens de prompt'] = st.get('tokens_prompt_mitjana')
        rows.append(r); idx.append(nom)
    return pd.DataFrame(rows, index=idx)

print('Referència manual (30 oracions, anotades a mà)'); display(taula('ref'))
print('Test (300 oracions, etiquetes del professor filtrades)'); display(taula('test'))

In [ ]:
#@title Gràfic per a la diapositiva
import matplotlib.pyplot as plt, numpy as np
t = taula('ref')
mets = ['exacte', 'F1_complements', 'subjecte', 'nucli_verbal', 'analisi_valida']
noms = {'exacte': 'anàlisi\nexacta', 'F1_complements': 'F1\ncomplements', 'subjecte': 'subjecte', 'nucli_verbal': 'nucli\nverbal', 'analisi_valida': 'sortida\nvàlida'}
cols = ['#E45756', '#54A24B', '#4C78A8']
fig, ax = plt.subplots(figsize=(11, 4.8)); w = 0.26; x = np.arange(len(mets))
for i, (nom, fila) in enumerate(t.iterrows()):
    b = ax.bar(x + (i - 1) * w, [fila[m] for m in mets], w, label=f"{nom}  ·  {fila['ms/frase']:.0f} ms/frase", color=cols[i])
    ax.bar_label(b, fmt='%.0f', fontsize=9, padding=2)
ax.set_xticks(x, [noms[m] for m in mets]); ax.set_ylim(0, 110); ax.set_ylabel('%')
ax.set_title('Referència manual · 30 oracions', loc='left', fontsize=14, fontweight='bold')
for s in ['top', 'right']: ax.spines[s].set_visible(False)
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3)
plt.tight_layout(); plt.savefig(f'{BASE}/resultats.png', dpi=200); plt.show()

In [ ]:
#@title Comparació visual a la referència manual
base_r = {r['frase']: r['analisi'] for r in llegeix('/content/ref_alumne_base.jsonl')}
dest_r = {r['frase']: r['analisi'] for r in llegeix('/content/ref_alumne_destillat.jsonl')}
for f, a in REFERENCIA:
    display(HTML(f"<hr><b>Referència</b>{mostra_html(f, a)}<b>Base</b>{mostra_html(f, base_r[f])}<b>Destil·lat</b>{mostra_html(f, dest_r[f])}"))

# 08 · Demo en directe
### *Escriu una oració, sense cap instrucció, i el destil·lat respon amb el JSON*

---

In [ ]:
#@title Carrega el model destil·lat
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
tok_d = AutoTokenizer.from_pretrained(MODEL_OUT)
try:
    mod_d = AutoModelForCausalLM.from_pretrained(MODEL_OUT, torch_dtype=torch.bfloat16).cuda().eval()
except Exception:
    from transformers import AutoModelForImageTextToText
    mod_d = AutoModelForImageTextToText.from_pretrained(MODEL_OUT, torch_dtype=torch.bfloat16).cuda().eval()

def analitza(frase):
    ids = tok_d(render(tok_d, frase, False, curt=True), return_tensors='pt').to('cuda')
    t0 = time.time()
    out = mod_d.generate(**ids, max_new_tokens=400, do_sample=False)
    txt = tok_d.decode(out[0, ids['input_ids'].shape[1]:], skip_special_tokens=True)
    a = extreu_json(txt)
    display(HTML(mostra_html(frase, a) + f"<small>{1000*(time.time()-t0):.0f} ms</small>"))
    print(json.dumps(a, ensure_ascii=False, indent=1) if a else txt)
    return a

In [ ]:
#@title Analitza una oració
frase = 'La Laia va regalar un dibuix a la seva àvia per Nadal.'  #@param {type:"string"}
analitza(frase);

# 09 · Per endur-se'l
### *(Opcional) Exportar a GGUF per fer-lo córrer al portàtil*

---

In [ ]:
#@title Exporta a GGUF (descomenta per executar)
# !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
# !pip install -q -r /content/llama.cpp/requirements.txt
# !python /content/llama.cpp/convert_hf_to_gguf.py {MODEL_OUT} --outtype q8_0 --outfile {BASE}/alumne_destillat_q8.gguf